# Colab 训练、评估与分析

这个 notebook 覆盖离线处理、训练、评估、分析和结果打包。
如果本地改了 `raman`，先上传最新的 `raman.zip`，再运行“解压库代码”和“刷新模块”单元。


In [ ]:
from pathlib import Path
import shutil

# 如果已经把 zip 上传到当前目录，就先在这里解压覆盖库代码。
PROJECT_ROOT = Path.cwd()
RAMAN_ZIP_PATH = PROJECT_ROOT / "raman.zip"
RAMAN_DIR = PROJECT_ROOT / "raman"

# 默认检测到 zip 时就执行一次解压。
RUN_UNPACK_RAMAN = RAMAN_ZIP_PATH.exists()
# 若想彻底覆盖旧库，先删除原目录再解压。
CLEAR_RAMAN_BEFORE_UNPACK = True

def unpack_library(zip_path, target_dir, should_unpack, clear_before_unpack):
    if should_unpack:
        if not zip_path.exists():
            raise FileNotFoundError(f"找不到压缩包：{zip_path}")
        if clear_before_unpack and target_dir.exists():
            shutil.rmtree(target_dir)
        shutil.unpack_archive(zip_path, PROJECT_ROOT)
        print(f"unpacked {zip_path.name} -> {target_dir}")
    else:
        print(f"skip unzip: {zip_path} not found")

unpack_library(
    RAMAN_ZIP_PATH,
    RAMAN_DIR,
    RUN_UNPACK_RAMAN,
    CLEAR_RAMAN_BEFORE_UNPACK,
)


In [ ]:
import importlib
import sys
# 修改库代码后，先从 sys.modules 清掉 raman，再重新导入
# 这种方式能处理已经删除或改名的旧模块
def reload_all():
    for name in list(sys.modules):
        if name == "raman" or name.startswith("raman."):
            del sys.modules[name]
    import raman.config as config_module
    print("Modules reloaded.")
    return config_module.config

config = reload_all()


In [ ]:
from dataclasses import asdict, is_dataclass


def _config_group_to_dict(group):
    if group is None:
        return {}
    if is_dataclass(group):
        return asdict(group)
    if hasattr(group, "to_dict"):
        return group.to_dict()
    return {
        key: value
        for key, value in vars(group).items()
        if not key.startswith("_")
    }


def _print_config_section(title, data):
    print(f"\n===== {title} =====")
    if not data:
        print("  <empty>")
        return
    for key in data:
        print(f"  {key}: {data[key]}")


_print_config_section(
    "Derived Values",
    {
        "dataset_root": getattr(config, "dataset_root", None),
        "in_channels": getattr(config, "in_channels", None),
        "delta": getattr(config, "delta", None),
    },
)
_print_config_section("Shared Input Config", _config_group_to_dict(getattr(config, "shared", None)))
_print_config_section("Model Run Config", _config_group_to_dict(getattr(config, "model", None)))
_print_config_section("Runtime Config", _config_group_to_dict(getattr(config, "runtime", None)))
_print_config_section(
    "Level Settings",
    {
        "train_per_parent": getattr(config, "train_per_parent", False),
        "use_align_loss": f"{getattr(config, 'use_align_loss', True)} (weight={getattr(config, 'align_loss_weight', 0.05)})",
        "use_supcon_loss": f"{getattr(config, 'use_supcon_loss', True)} (weight={getattr(config, 'supcon_loss_weight', 0.03)}, tau={getattr(config, 'supcon_tau', 0.15)})",
    },
)


In [ ]:
from pathlib import Path
import shutil
import sys

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from raman.data.profiles import get_dataset_dir, get_profile

# 在这里切换数据集，建议用英文别名。
DATASET_NAME = "MICRO"  #  MICRO / ENT / NENT / GP / FUNG

profile = get_profile(DATASET_NAME)
# 解析 dataset/ 下的目录结构。
dataset_dir = get_dataset_dir(profile, PROJECT_ROOT)
dataset_dir.mkdir(parents=True, exist_ok=True)
# dataset_root / bad_bands 会随 dataset_name 联动
config.dataset_name = DATASET_NAME
pca_log_path = dataset_dir / profile.pca_log_name
cosmic_log_path = dataset_dir / profile.cosmic_ray_log_name

print("dataset_name =", config.dataset_name)
print("dataset_dir =", dataset_dir)
print("dataset_root =", config.dataset_root)
print("bad_bands =", config.bad_bands)
print("pca_log =", pca_log_path)
print("cosmic_ray_log =", cosmic_log_path)


In [ ]:
from raman.data.archive import unpack_init

# 先把 init.npz 手动上传到当前数据集目录下，再执行这里的解压或打包。
pack_path = dataset_dir / profile.root_init_pack
init_dir = dataset_dir / profile.root_init

# init.npz 已经在 dataset 目录下时，默认执行一次解压。
RUN_UNPACK_INIT = pack_path.exists()
# 若要覆盖旧的 init，可以先清空再解压。
CLEAR_INIT_BEFORE_UNPACK = True

# 先解压 init.npz。
if RUN_UNPACK_INIT:
    if not pack_path.exists():
        raise FileNotFoundError(f"找不到打包文件：{pack_path}")
    if CLEAR_INIT_BEFORE_UNPACK and init_dir.exists():
        shutil.rmtree(init_dir)
        init_dir.mkdir(parents=True, exist_ok=True)
    unpack_init(pack_path, init_dir)

print(f"packed file = {pack_path}")
print(f"init = {init_dir}")


In [ ]:
from raman.data.build import build_test, build_train

# 这一段从 init 处理到 train / test
RUN_BUILD_TRAIN = True
RUN_BUILD_TEST = False
# 只有想清空旧结果重跑时才打开
REBUILD_TRAIN = True
REBUILD_TEST = False

train_dir = dataset_dir / profile.root_train_clean
train_fig_dir = dataset_dir / profile.root_train_fig
test_dir = dataset_dir / profile.root_test_clean
test_fig_dir = dataset_dir / profile.root_test_fig
pca_log_path = dataset_dir / profile.pca_log_name
cosmic_log_path = dataset_dir / profile.cosmic_ray_log_name

# 重建目录时使用，避免旧结果和新结果混在一起
def reset_dir(path):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

# build_train 直接从 init 生成 train，并执行 PCA 过滤
if RUN_BUILD_TRAIN:
    if REBUILD_TRAIN:
        reset_dir(train_dir)
        reset_dir(train_fig_dir)
    build_train(profile, dataset_dir)

# 如果需要生成 test，再打开 RUN_BUILD_TEST
if RUN_BUILD_TEST:
    if REBUILD_TEST:
        reset_dir(test_dir)
        reset_dir(test_fig_dir)
    build_test(profile, dataset_dir)

print("Done.")
print(f"train = {train_dir}")
print(f"fig_train = {train_fig_dir}")
print(f"test = {test_dir}")
print(f"fig_test = {test_fig_dir}")
print(f"pca_log = {pca_log_path}")
print(f"cosmic_ray_log = {cosmic_log_path}")


In [ ]:
from raman.data.count import count_dataset, print_results

# 默认统计 train，也可以改成 test。
COUNT_SUBDIR = "train"
target_dir = dataset_dir / COUNT_SUBDIR

tree, total_files = count_dataset(target_dir)
print_results(tree, total_files)


In [ ]:
# 训练配置
CURRENT_TRAIN_LEVEL = "level_1"
TRAIN_ONLY_PARENT_NAME = None
TRAIN_ONLY_PARENT = None

OVERRIDE_ALIGN_LOSS_WEIGHT = None
OVERRIDE_SUPCON_TAU = None
OVERRIDE_SUPCON_LOSS_WEIGHT = None
OVERRIDE_SEED = None
OVERRIDE_SPLIT_BY_SOURCE_PREFIX = None

# 训练轮次和早停轮次；None 表示使用 raman.config.py 里的默认值。
OVERRIDE_EPOCHS = None  # 例如 60
OVERRIDE_PATIENCE = None  # 例如 20

SUPCON_START_OVERRIDE = None
SUPCON_END_OVERRIDE = None
ALIGN_START_OVERRIDE = None
ALIGN_END_OVERRIDE = None

OVERRIDE_OUTPUT_DIR = None
# 如果要把结果写进已有实验目录，就改 OVERRIDE_OUTPUT_DIR。

if OVERRIDE_EPOCHS is not None:
    config.epochs = int(OVERRIDE_EPOCHS)
    config.scheduler_Tmax = int(config.epochs)
if OVERRIDE_PATIENCE is not None:
    config.patience = int(OVERRIDE_PATIENCE)

# 只有需要临时改 SupCon 节奏时，才设置这些 override。
if SUPCON_START_OVERRIDE is not None:
    config.supcon_start = int(SUPCON_START_OVERRIDE)
if SUPCON_END_OVERRIDE is not None:
    config.supcon_end = int(SUPCON_END_OVERRIDE)
if ALIGN_START_OVERRIDE is not None:
    config.align_start = int(ALIGN_START_OVERRIDE)
if ALIGN_END_OVERRIDE is not None:
    config.align_end = int(ALIGN_END_OVERRIDE)
if OVERRIDE_SEED is not None:
    config.seed = int(OVERRIDE_SEED)
if OVERRIDE_SPLIT_BY_SOURCE_PREFIX is not None:
    config.split_by_source_prefix = bool(OVERRIDE_SPLIT_BY_SOURCE_PREFIX)

print("current_train_level =", CURRENT_TRAIN_LEVEL)
print("epochs =", config.epochs)
print("patience =", config.patience)
print("scheduler_Tmax =", config.scheduler_Tmax)
print("supcon_start/end =", config.supcon_start, config.supcon_end)
print("align_start/end =", config.align_start, config.align_end)
print("seed =", config.seed)
print("split_by_source_prefix =", config.split_by_source_prefix)
print("dataset_root =", config.dataset_root)


In [ ]:
from raman.trainer import TrainOverrides, run_training

# 训练完成后会返回实验根目录，后面的评估和打包都直接复用这个目录
train_result = run_training(
    config,
    overrides=TrainOverrides(
        current_train_level=CURRENT_TRAIN_LEVEL,
        train_only_parent_name=TRAIN_ONLY_PARENT_NAME,
        train_only_parent=TRAIN_ONLY_PARENT,
        override_align_loss_weight=OVERRIDE_ALIGN_LOSS_WEIGHT,
        override_supcon_tau=OVERRIDE_SUPCON_TAU,
        override_supcon_loss_weight=OVERRIDE_SUPCON_LOSS_WEIGHT,
        override_output_dir=OVERRIDE_OUTPUT_DIR,
    ),
)

EXP_DIR = train_result["output_dir"]
print("EXP_DIR =", EXP_DIR)
print("run_dirs =", train_result.get("run_dirs", []))
train_result


## 结果通用配置


In [ ]:
# 结果入口通用配置
# 如果跳过训练、直接检查已有实验，就在这里手动填实验根目录
MANUAL_EXP_DIR = None

if MANUAL_EXP_DIR is not None:
    EXP_DIR = MANUAL_EXP_DIR
elif "EXP_DIR" not in globals():
    EXP_DIR = None

# baseline 参数
BASELINE_USE_ALL_CHANNELS = False
BASELINE_PCA_N_COMPONENTS = 5
BASELINE_SVM_C = 1.0
BASELINE_SVM_KERNEL = "rbf"
BASELINE_SVM_GAMMA = "scale"
BASELINE_RANDOM_STATE = 42

# analysis 参数
INHERIT_MISSING_LEVELS = False
ANALYSIS_HEATMAP_NUM_BATCHES = 10
ANALYSIS_HEATMAP_STEPS = 32
ANALYSIS_HEATMAP_MAX_PER_CLASS = 50
ANALYSIS_HEATMAP_ROW_NORM = "max"
ANALYSIS_HEATMAP_USE_TRAIN_LOADER = True
ANALYSIS_HEATMAP_TOPK_PER_CLASS = 5
ANALYSIS_SHOW_MAX_IMAGES = 12

# 混淆矩阵展示缩放，None 表示按生成图片原始尺寸展示
# 生成图片尺寸会按类别数量自动放大，这里只控制 notebook 里的显示缩放
CONFUSION_MATRIX_WIDTH = None
CONFUSION_MATRIX_HEIGHT = None

import json
import os
import shutil
from pathlib import Path
from IPython.display import Image, display

from raman.analysis import HeatmapConfig


def baseline_kwargs():
    return {
        "use_all_channels": BASELINE_USE_ALL_CHANNELS,
        "pca_n_components": BASELINE_PCA_N_COMPONENTS,
        "svm_c": BASELINE_SVM_C,
        "svm_kernel": BASELINE_SVM_KERNEL,
        "svm_gamma": BASELINE_SVM_GAMMA,
        "random_state": BASELINE_RANDOM_STATE,
    }


def heatmap_config():
    return HeatmapConfig(
        num_batches=ANALYSIS_HEATMAP_NUM_BATCHES,
        steps=ANALYSIS_HEATMAP_STEPS,
        max_per_class=ANALYSIS_HEATMAP_MAX_PER_CLASS,
        row_norm=ANALYSIS_HEATMAP_ROW_NORM,
        use_train_loader=ANALYSIS_HEATMAP_USE_TRAIN_LOADER,
        topk_per_class=ANALYSIS_HEATMAP_TOPK_PER_CLASS,
    )


def clear_run_selection():
    os.environ.pop("RAMAN_RUN_SELECTION", None)


def apply_run_selection(run_selection):
    if run_selection:
        os.environ["RAMAN_RUN_SELECTION"] = json.dumps(run_selection, ensure_ascii=False)
        print("RAMAN_RUN_SELECTION =", os.environ["RAMAN_RUN_SELECTION"])
    else:
        clear_run_selection()
        print("RAMAN_RUN_SELECTION cleared")


def _has_run_or_best(slot_dir):
    if (slot_dir / "best").exists():
        return True
    return any(path.is_dir() and path.name.startswith("run_") for path in slot_dir.iterdir())


def _model_slot_dirs(exp_dir):
    root = Path(exp_dir)
    if not root.exists():
        return []
    slots = []
    for level_dir in sorted(root.glob("level_*")):
        if not level_dir.is_dir():
            continue
        if _has_run_or_best(level_dir):
            slots.append(level_dir)
        for child_dir in sorted(level_dir.iterdir()):
            if child_dir.is_dir() and child_dir.name.startswith(level_dir.name + "_"):
                if _has_run_or_best(child_dir):
                    slots.append(child_dir)
    return slots


def list_run_slots(exp_dir):
    root = Path(exp_dir)
    print("\n[Run 列表]")
    for slot_dir in _model_slot_dirs(root):
        key = slot_dir.relative_to(root).as_posix()
        runs = sorted(path.name for path in slot_dir.iterdir() if path.is_dir() and path.name.startswith("run_"))
        best_dir = slot_dir / "best"
        best_runs = []
        if best_dir.exists():
            best_runs = sorted(path.name for path in best_dir.iterdir() if path.is_dir() and path.name.startswith("run_"))
        print(f"  {key}")
        print(f"    runs: {runs}")
        print(f"    best: {best_runs}")


def auto_single_run_dir():
    if "train_result" not in globals() or EXP_DIR is None:
        return None
    run_dirs = train_result.get("run_dirs", [])
    if len(run_dirs) != 1:
        return None
    return str(Path(EXP_DIR) / run_dirs[0])


def show_confusion_matrix(result_dir, filename="confusion_matrix.png", width=None, height=None):
    result_dir = Path(result_dir)
    cm_path = result_dir / filename
    if not cm_path.exists():
        raise FileNotFoundError(f"找不到混淆矩阵：{cm_path}")
    width = CONFUSION_MATRIX_WIDTH if width is None else width
    height = CONFUSION_MATRIX_HEIGHT if height is None else height
    image_kwargs = {}
    if width is not None:
        image_kwargs["width"] = width
    if height is not None:
        image_kwargs["height"] = height
    display(Image(filename=str(cm_path), **image_kwargs))
    print("confusion_matrix =", cm_path)


def show_analysis_figures(result_dir, max_images=None):
    result_dir = Path(result_dir)
    fig_dir = result_dir / "figures"
    if not fig_dir.exists():
        raise FileNotFoundError(f"找不到 analysis figures 目录：{fig_dir}")
    pngs = sorted(fig_dir.glob("*.png"))
    if not pngs:
        raise FileNotFoundError(f"analysis figures 目录下没有 png：{fig_dir}")
    max_images = ANALYSIS_SHOW_MAX_IMAGES if max_images is None else max_images
    for image_path in pngs[:max_images]:
        print(image_path.name)
        display(Image(filename=str(image_path)))
    if len(pngs) > max_images:
        print(f"还有 {len(pngs) - max_images} 张图未展示，目录：{fig_dir}")


EXPERIMENT_ROOT_PACKAGE_FILES = [
    "class_names.json",
    "hierarchy_meta.json",
    "shared_config.yaml",
    "train_split.json",
    "val_split.json",
]


def package_directory(source_dir, output_dir="/content", package_name=None):
    source_dir = Path(source_dir)
    if not source_dir.exists() or not source_dir.is_dir():
        raise FileNotFoundError(f"找不到要压缩的目录：{source_dir}")
    package_name = package_name or source_dir.name
    out_base = Path(output_dir) / package_name
    zip_path = shutil.make_archive(
        base_name=str(out_base),
        format="zip",
        root_dir=str(source_dir.parent),
        base_dir=source_dir.name,
    )
    print("package_source =", source_dir)
    print("package_zip =", zip_path)
    return zip_path


if EXP_DIR is not None:
    print("EXP_DIR =", EXP_DIR)
    list_run_slots(EXP_DIR)
clear_run_selection()


## 单模型检查


In [ ]:
# 单模型配置
# 填具体 run_* 或 best/run_*；None 时，如果刚训练只产生了一个 run，会自动使用它
SINGLE_RUN_DIR = None
SINGLE_LEVEL = "level_1"
SINGLE_PARENT_IDX = None

if SINGLE_RUN_DIR is None:
    SINGLE_RUN_DIR = auto_single_run_dir()
if SINGLE_RUN_DIR is None:
    raise ValueError("请填写 SINGLE_RUN_DIR，或先训练一个只产生单个 run 的任务")

clear_run_selection()
LAST_RESULT_DIR = SINGLE_RUN_DIR
print("SINGLE_RUN_DIR =", SINGLE_RUN_DIR)


In [ ]:
# 单模型 eval 结果和混淆矩阵展示
from raman.eval import run_eval_single_model

single_val_result_dir = run_eval_single_model(SINGLE_RUN_DIR, level=SINGLE_LEVEL)
LAST_RESULT_DIR = single_val_result_dir
print("single_val_result_dir =", single_val_result_dir)
show_confusion_matrix(single_val_result_dir)


In [ ]:
# 单模型 baseline 计算和混淆矩阵展示
from raman.eval import run_baseline_single_model

single_baseline_result_dir = run_baseline_single_model(
    SINGLE_RUN_DIR,
    level=SINGLE_LEVEL,
    **baseline_kwargs(),
)
LAST_RESULT_DIR = single_baseline_result_dir
print("single_baseline_result_dir =", single_baseline_result_dir)
show_confusion_matrix(single_baseline_result_dir)


In [ ]:
# 单模型 analysis 计算和图展示
from raman.analysis import run_analysis_single_model

single_analysis_result_dir = run_analysis_single_model(
    SINGLE_RUN_DIR,
    level=SINGLE_LEVEL,
    parent_idx=SINGLE_PARENT_IDX,
    inherit_missing_levels=INHERIT_MISSING_LEVELS,
    heatmap_cfg=heatmap_config(),
)
LAST_RESULT_DIR = single_analysis_result_dir
print("single_analysis_result_dir =", single_analysis_result_dir)
show_analysis_figures(single_analysis_result_dir)


In [ ]:
# 压缩单 run 文件夹
SINGLE_PACKAGE_OUTPUT_DIR = "/content"
SINGLE_PACKAGE_NAME = None

single_run_zip = package_directory(
    SINGLE_RUN_DIR,
    output_dir=SINGLE_PACKAGE_OUTPUT_DIR,
    package_name=SINGLE_PACKAGE_NAME,
)


In [ ]:
# 单独压缩实验根目录必要文件
# 包含 class_names.json、hierarchy_meta.json、shared_config.yaml、train_split.json、val_split.json
ROOT_FILES_EXP_DIR = EXP_DIR
ROOT_FILES_PACKAGE_OUTPUT_DIR = "/content"
ROOT_FILES_PACKAGE_NAME = None

if ROOT_FILES_EXP_DIR is None:
    raise ValueError("请填写 ROOT_FILES_EXP_DIR，或先运行训练得到 EXP_DIR")

exp_root = Path(ROOT_FILES_EXP_DIR).resolve()
if not exp_root.exists() or not exp_root.is_dir():
    raise FileNotFoundError(f"找不到实验根目录：{exp_root}")

package_name = ROOT_FILES_PACKAGE_NAME or f"{exp_root.name}_root_files"
stage_dir = Path(ROOT_FILES_PACKAGE_OUTPUT_DIR) / f"_{package_name}_stage"
if stage_dir.exists():
    shutil.rmtree(stage_dir)
stage_dir.mkdir(parents=True, exist_ok=True)

copied_root_files = []
for file_name in EXPERIMENT_ROOT_PACKAGE_FILES:
    src_file = exp_root / file_name
    if not src_file.exists():
        raise FileNotFoundError(f"实验根目录缺少打包文件：{src_file}")
    shutil.copy2(src_file, stage_dir / file_name)
    copied_root_files.append(file_name)

root_files_zip = shutil.make_archive(
    base_name=str(Path(ROOT_FILES_PACKAGE_OUTPUT_DIR) / package_name),
    format="zip",
    root_dir=str(stage_dir),
)
shutil.rmtree(stage_dir)
print("experiment_root =", exp_root)
print("copied_root_files =", copied_root_files)
print("package_zip =", root_files_zip)


## 单层多模型诊断

这里使用真实父类路由，适合看目标层子模型本身的能力。单子类分支会直通，例如 FUNG -> Candida 会显示为 100%，不建议把这张图作为端到端展示图。


In [ ]:
# 单层多模型配置
# 单层多模型默认使用 best/ 中唯一 run；RUN_SELECTION 只在这里临时指定，不会创建或修改 best/
# 这里按真实父类分发，单子类 parent 会直通，因此它是诊断图，不是端到端展示图
LEVEL_ONLY_EXP_DIR = EXP_DIR
LEVEL_ONLY_TARGET_LEVEL = "level_2"
LEVEL_ONLY_PARENT_IDX = "all"
RUN_SELECTION = {}
# 示例：{"level_2/level_2_0": "run_20260524_064200"}

if LEVEL_ONLY_EXP_DIR is None:
    raise ValueError("请填写 LEVEL_ONLY_EXP_DIR，或先运行训练得到 EXP_DIR")

apply_run_selection(RUN_SELECTION)
level_only_root = Path(LEVEL_ONLY_EXP_DIR) / LEVEL_ONLY_TARGET_LEVEL / "level_only_result"
LAST_RESULT_DIR = str(level_only_root)
print("level_only_root =", level_only_root)


In [ ]:
# 单层多模型 eval 结果和混淆矩阵展示（真实父类路由诊断图）
from raman.eval import run_eval_level_only

level_only_val_result_dir = run_eval_level_only(LEVEL_ONLY_EXP_DIR, LEVEL_ONLY_TARGET_LEVEL)
LAST_RESULT_DIR = level_only_val_result_dir
print("level_only_val_result_dir =", level_only_val_result_dir)
show_confusion_matrix(level_only_val_result_dir)


In [ ]:
# 单层多模型 baseline 计算和混淆矩阵展示
from raman.eval import run_baseline_level_only

level_only_baseline_result_dir = run_baseline_level_only(
    LEVEL_ONLY_EXP_DIR,
    LEVEL_ONLY_TARGET_LEVEL,
    **baseline_kwargs(),
)
LAST_RESULT_DIR = level_only_baseline_result_dir
print("level_only_baseline_result_dir =", level_only_baseline_result_dir)
show_confusion_matrix(level_only_baseline_result_dir)


In [ ]:
# 单层多模型 analysis 计算和图展示
from raman.analysis import run_analysis_level_only

level_only_analysis_result_dir = run_analysis_level_only(
    LEVEL_ONLY_EXP_DIR,
    LEVEL_ONLY_TARGET_LEVEL,
    parent_idx=LEVEL_ONLY_PARENT_IDX,
    inherit_missing_levels=INHERIT_MISSING_LEVELS,
    heatmap_cfg=heatmap_config(),
)
LAST_RESULT_DIR = level_only_analysis_result_dir
print("level_only_analysis_result_dir =", level_only_analysis_result_dir)
show_analysis_figures(level_only_analysis_result_dir)


In [ ]:
# 压缩选择目录
# 默认压缩 level_only_result；也可以手动改成某个结果目录
PACKAGE_SOURCE_DIR = level_only_root
PACKAGE_OUTPUT_DIR = "/content"
PACKAGE_NAME = None

selected_zip = package_directory(
    PACKAGE_SOURCE_DIR,
    output_dir=PACKAGE_OUTPUT_DIR,
    package_name=PACKAGE_NAME,
)


## 多层多模型级联展示

这里先调用上一层模型预测，再进入目标层。用于展示 level_2/level_3 更真实：FUNG -> Candida 这类单子类分支会继承 level_1 的预测错误，不会因为真实父类路由而固定 100%。


In [ ]:
# 多层多模型级联配置
# 多层级联默认使用各层 best/ 中唯一 run，不使用 RUN_SELECTION
# 展示 level_2 时建议用这一组结果，而不是单层多模型诊断图
CASCADE_EXP_DIR = EXP_DIR
CASCADE_TARGET_LEVEL = "level_2"
CASCADE_PARENT_IDX = "all"

if CASCADE_EXP_DIR is None:
    raise ValueError("请填写 CASCADE_EXP_DIR，或先运行训练得到 EXP_DIR")

clear_run_selection()
cascade_root = Path(CASCADE_EXP_DIR) / CASCADE_TARGET_LEVEL / "cascade_result"
LAST_RESULT_DIR = str(cascade_root)
print("cascade_root =", cascade_root)


In [ ]:
# 多层多模型 eval 结果和混淆矩阵展示（继承上一层预测，适合展示）
from raman.eval import run_eval_cascade

cascade_val_result_dir = run_eval_cascade(CASCADE_EXP_DIR, CASCADE_TARGET_LEVEL)
LAST_RESULT_DIR = cascade_val_result_dir
print("cascade_val_result_dir =", cascade_val_result_dir)
show_confusion_matrix(cascade_val_result_dir)


In [ ]:
# 多层多模型 analysis 计算和图展示
from raman.analysis import run_analysis_cascade

cascade_analysis_result_dir = run_analysis_cascade(
    CASCADE_EXP_DIR,
    CASCADE_TARGET_LEVEL,
    parent_idx=CASCADE_PARENT_IDX,
    inherit_missing_levels=INHERIT_MISSING_LEVELS,
    heatmap_cfg=heatmap_config(),
)
LAST_RESULT_DIR = cascade_analysis_result_dir
print("cascade_analysis_result_dir =", cascade_analysis_result_dir)
show_analysis_figures(cascade_analysis_result_dir)


In [ ]:
# 最后压缩选择目录
# 默认压缩最近一次结果目录；也可以手动改成 run_*、level_only_result、cascade_result 或任意结果目录
FINAL_PACKAGE_SOURCE_DIR = globals().get("LAST_RESULT_DIR", EXP_DIR)
FINAL_PACKAGE_OUTPUT_DIR = "/content"
FINAL_PACKAGE_NAME = None

final_zip = package_directory(
    FINAL_PACKAGE_SOURCE_DIR,
    output_dir=FINAL_PACKAGE_OUTPUT_DIR,
    package_name=FINAL_PACKAGE_NAME,
)
